# Dataset claim checks (report Chapter 3, Table of external datasets)

Checks every number the report states about the seven external datasets against the files on your own machine. Nothing is trained and nothing is written to the repo except one small report file.

**Built for big files.** CSVs are read in chunks, the Hugging Face dataset is read from its metadata (and optionally streamed), and PDFs are read one at a time. Nothing loads a whole large file into memory.

**Before you run**
1. Put each dataset in its own folder under `DATA_DIR` (default `./data`, or set the `CARNET_DATA` environment variable). Folder names do not need renaming: each one is found by a keyword (linkedin, success, investment, job matching, resume dataset pdf, screen, roman). Keep these folders out of git.
2. `pip install pandas pyarrow pypdf datasets` (`datasets` is only needed if there is no local Roman-Urdu folder) (Python 3.11 `.venv`, ask before installing).
3. Run all cells. Each check prints PASS, FAIL or SKIP (SKIP means the file was not found). The last cell writes `dataset_claim_check_report.md` next to the notebook.

A FAIL does not mean the dataset is wrong. It means the report and the file disagree. Decide which to fix and log it in `docs/DECISIONS.md` and `docs/pending-report-changes.md`.

In [1]:
import os
os.environ["CARNET_DATA"] = r"E:\Documents\advanced_statistics\fyp\founders-carnet\carnet_data\data"


In [3]:
import os, re, json, random, warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore")
DATA_DIR = Path(os.environ.get("CARNET_DATA", "./data")).resolve()
CHUNK = 200_000          # rows per chunk for big CSVs
PDF_SAMPLE = 30          # report uses a 30-file sample; set to None to scan every PDF (slow)

FOLDERS = {
    "roman_urdu_parl": "Roman-Urdu-Parl split (local copy of the Hugging Face repo; online metadata is the fallback)",
    "linkedin": "LinkedIn-Dataset (git clone of navid-aub/LinkedIn-Dataset)",
    "crunchbase_success": "Kaggle yanmaksi/big-startup-secsees-fail-dataset-from-crunchbase",
    "crunchbase_invest": "Kaggle arindam235/startup-investments-crunchbase",
    "hse": "Kaggle darysha/hse-hackathon",
    "resume_pdf": "Kaggle hadikp/resume-data-pdf",
    "ats": "Kaggle amaymishra11/resume-screening-dataset-ats-and-nlp-ready",
}
# Folder names do not have to match. Each dataset is found by a keyword in its folder name
# (case-insensitive), so "LinkedIn-Dataset-main" and "Resume-Job Matching Dataset" work as they are.
KEYWORDS = {
    "roman_urdu_parl": ["roman"],
    "linkedin": ["linkedin"],
    "crunchbase_success": ["success"],
    "crunchbase_invest": ["investment"],
    "hse": ["job matching", "resume-job", "hse"],
    "resume_pdf": ["resume dataset pdf", "resume-data-pdf"],
    "ats": ["screen", "ats"],
}
def resolve(key):
    # returns the folder for a dataset key, or a path that does not exist
    if DATA_DIR.exists():
        for d in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
            if any(w in d.name.lower() for w in KEYWORDS[key]):
                return d
    return DATA_DIR / key

print("DATA_DIR =", DATA_DIR)
for k, v in FOLDERS.items():
    p = resolve(k)
    print(f"{'found  ' if p.exists() else 'MISSING'} {k:20s} -> {p.name if p.exists() else '(none)'}")

RESULTS = []   # one row per claim

def check(dataset, claim, expected, observed, tol=0):
    """Record one claim. Numbers match within tol; everything else must be equal."""
    try:
        ok = abs(float(observed) - float(expected)) <= tol
    except (TypeError, ValueError):
        ok = observed == expected
    RESULTS.append({"dataset": dataset, "claim": claim, "expected": expected, "observed": observed,
                    "result": "PASS" if ok else "FAIL"})
    print(f"{'PASS' if ok else 'FAIL'}  {claim}: report {expected!r}, file {observed!r}")

def skip(dataset, claim, why):
    RESULTS.append({"dataset": dataset, "claim": claim, "expected": "", "observed": why, "result": "SKIP"})
    print(f"SKIP  {claim}: {why}")

def info(dataset, claim, observed):
    for r in RESULTS:
        if r['dataset'] == dataset and r['claim'] == claim:
            r['observed'] = observed
            print(f"INFO  {claim}: {observed}")
            return
    RESULTS.append({"dataset": dataset, "claim": claim, "expected": "(no number in report)",
                    "observed": observed, "result": "INFO"})
    print(f"INFO  {claim}: {observed}")

def find_file(folder, pattern):
    d = resolve(folder)
    if not d.exists():
        return None
    hits = sorted(d.rglob(pattern))
    return hits[0] if hits else None

def read_csv_chunks(path, **kw):
    for enc in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(path, chunksize=CHUNK, low_memory=False, encoding=enc, **kw), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"could not decode {path} with any tried encoding")

PERSONAL = re.compile(r"(^|_|\b)(name|first.?name|last.?name|age|gender|sex|birth|dob|email|phone|mobile|contact|address)(\b|_|$)", re.I)
def personal_columns(dataset, cols):
    hits = [c for c in cols if PERSONAL.search(str(c).strip())]
    info(dataset, "columns that look like personal fields (report: excluded, never used)", hits or "none")

DATA_DIR = E:\Documents\advanced_statistics\fyp\founders-carnet\carnet_data\data
found   roman_urdu_parl      -> roman_urdu_parl_split_dataset
found   linkedin             -> LinkedIn-Dataset-main
found   crunchbase_success   -> Startup SuccessFail Dataset from Crunchbase
found   crunchbase_invest    -> StartUp Investments (Crunchbase)
found   hse                  -> Resume-Job Matching Dataset
found   resume_pdf           -> Resume Dataset PDF
found   ats                  -> How Companies Screen Resumes ATS Dataset (NLP+ML)


## 1. Roman-Urdu-Parl split
Report: 6,374,678 pairs (6,333,218 train, 20,719 validation, 20,741 test); two text columns; the original paper reports 6,365,808.

In [4]:
D = "Roman-Urdu-Parl"
root = resolve("roman_urdu_parl")
EXP = {"train": 6_333_218, "validation": 20_719, "test": 20_741}
COLS = ["Urdu text", "Roman-Urdu text"]

def split_of(path):
    s = str(path).lower()
    for name in ("validation", "valid", "val", "test", "train"):
        if name in s:
            return "validation" if name in ("valid", "val") else name
    return "unknown"

splits, cols = {}, None
if root.exists():
    # local copy of the Hugging Face repo: row counts come from file footers, nothing is loaded
    import pyarrow.parquet as pq
    pqs = sorted(root.rglob("*.parquet"))
    for f in pqs:
        pf = pq.ParquetFile(f)
        splits[split_of(f.relative_to(root))] = splits.get(split_of(f.relative_to(root)), 0) + pf.metadata.num_rows
        cols = cols or pf.schema_arrow.names
    if not pqs:
        for f in sorted(list(root.rglob("*.csv")) + list(root.rglob("*.jsonl")) + list(root.rglob("*.json"))):
            if f.name.lower() in ("dataset_info.json", "state.json"):
                continue
            n = 0
            if f.suffix == ".csv":
                for ch in pd.read_csv(f, chunksize=CHUNK):
                    n += len(ch); cols = cols or list(ch.columns)
            else:
                with open(f, encoding="utf-8") as fh:
                    n = sum(1 for _ in fh)
            splits[split_of(f.relative_to(root))] = splits.get(split_of(f.relative_to(root)), 0) + n
    info(D, "read from local folder", root.name)
else:
    try:
        from datasets import load_dataset_builder
        info_ = load_dataset_builder("Mavkif/Roman-Urdu-Parl-split").info   # metadata only
        splits = {k: int(v.num_examples) for k, v in (info_.splits or {}).items()}
        cols = list((info_.features or {}).keys())
        info(D, "read from Hugging Face metadata", "no local folder found")
    except Exception as e:
        skip(D, "Roman-Urdu-Parl checks", f"no local folder and Hugging Face failed: {type(e).__name__}")

if "train" not in splits and "unknown" in splits:
    splits["train"] = splits.pop("unknown")

if splits:
    info(D, "rows per split found", splits)
    for sp, exp in EXP.items():
        check(D, f"{sp} rows", exp, splits.get(sp))
    check(D, "total rows", 6_374_678, sum(splits.values()))
    info(D, "difference from the paper's 6,365,808", sum(splits.values()) - 6_365_808)
    check(D, "columns", COLS, cols)
    # quality of the first 5,000 rows of the smallest split (small)
    small = [f for f in (sorted(root.rglob("*.parquet")) if root.exists() else []) if split_of(f.relative_to(root)) == "validation"]
    if small:
        s = next(pq.ParquetFile(small[0]).iter_batches(batch_size=5000)).to_pandas()
        info(D, "empty text in first 5,000 validation rows",
             int(((s.iloc[:, 0].astype(str).str.strip() == "") | (s.iloc[:, 1].astype(str).str.strip() == "")).sum()))
        info(D, "duplicate pairs in first 5,000 validation rows", int(s.duplicated().sum()))

INFO  read from local folder: roman_urdu_parl_split_dataset
INFO  rows per split found: {'test': 20741, 'validation': 20719, 'train': 6365808}
FAIL  train rows: report 6333218, file 6365808
PASS  validation rows: report 20719, file 20719
PASS  test rows: report 20741, file 20741
FAIL  total rows: report 6374678, file 6407268
INFO  difference from the paper's 6,365,808: 41460
PASS  columns: report ['Urdu text', 'Roman-Urdu text'], file ['Urdu text', 'Roman-Urdu text']


## 2. LinkedIn-Dataset (Ayoobi et al.)
Report: 3,600 profiles, 39 columns: 1,800 legitimate, 600 fake, and 600 each generated by ChatGPT from legitimate and from fake profile statistics. Repository limits use to research.

In [5]:
D = "LinkedIn"
root = resolve("linkedin")
f = find_file("linkedin", "*.pcl") or find_file("linkedin", "*.pkl")
lic = [p.name for p in root.iterdir() if p.name.lower().startswith("licen")] if root.exists() else None
if root.exists():
    check(D, "LICENSE file present in repo (report: none stated)", False, bool(lic))
if f is None:
    skip(D, "profile counts", "no .pcl or .pkl file found")
else:
    obj = pd.read_pickle(f)   # file is about 25 MB
    info(D, "pickle type", type(obj).__name__)
    if isinstance(obj, pd.DataFrame):
        check(D, "profiles (rows)", 3600, obj.shape[0])
        check(D, "columns", 39, obj.shape[1])
        personal_columns(D, obj.columns)
        # the label column is the one whose value counts look like the four groups
        found = False
        for c in obj.columns:
            try:
                vc = obj[c].value_counts(dropna=False)
            except TypeError:
                continue
            if 2 <= len(vc) <= 8 and vc.sum() == 3600:
                info(D, f"candidate label column '{c}'", vc.to_dict())
                if sorted(vc.values.tolist()) == [600, 600, 600, 1800]:
                    check(D, "group sizes (1,800 + 600 + 600 + 600)", [600, 600, 600, 1800], sorted(vc.values.tolist()))
                    found = True
        if not found:
            skip(D, "group sizes", "no column splits the rows 1,800 / 600 / 600 / 600; read the dataset's README and set the label column by hand")
    else:
        info(D, "structure (not a DataFrame)", {k: (type(v).__name__, len(v) if hasattr(v, '__len__') else '') for k, v in list(obj.items())[:10]} if isinstance(obj, dict) else str(type(obj)))
        skip(D, "profile counts", "pickle is not a DataFrame; inspect the structure above")

PASS  LICENSE file present in repo (report: none stated): report False, file False
INFO  pickle type: DataFrame
PASS  profiles (rows): report 3600, file 3600
PASS  columns: report 39, file 39
INFO  columns that look like personal fields (report: excluded, never used): ['Full Name']
INFO  candidate label column 'Photo': {'Yes': 3062, 'No': 538}
INFO  candidate label column 'Number of Educations': {2: 1266, 1: 925, 3: 924, 0: 485}
INFO  candidate label column 'Number of Licenses': {0: 2581, 3: 478, 1: 349, 2: 192}
INFO  candidate label column 'Number of Volunteering': {0: 2607, 3: 424, 1: 320, 2: 249}
INFO  candidate label column 'Number of Skills': {3: 2762, 0: 762, 1: 57, 2: 19}
INFO  candidate label column 'Number of Recommendations': {0: 2152, 6: 471, 1: 347, 3: 175, 2: 169, 4: 163, 5: 123}
INFO  candidate label column 'Number of Projects': {0: 3343, 3: 108, 1: 106, 2: 43}
INFO  candidate label column 'Number of Publications': {0: 3226, 3: 210, 1: 110, 2: 54}
INFO  candidate label co

## 3. Startup Success/Fail (Crunchbase)
Report: 66,368 rows, 14 columns, with an operating status field.

In [6]:
D = "Crunchbase success/fail"
f = find_file("crunchbase_success", "*.csv")
if f is None:
    skip(D, "row and column counts", "no CSV found")
else:
    rows, cols, dup_seen, status = 0, None, 0, {}
    chunks, enc = read_csv_chunks(f)
    first = None
    for ch in chunks:
        cols = list(ch.columns); rows += len(ch)
        if "status" in ch.columns:
            for k, v in ch["status"].value_counts(dropna=False).items():
                status[k] = status.get(k, 0) + int(v)
    check(D, "rows", 66_368, rows)
    check(D, "columns", 14, len(cols))
    check(D, "has a status field", True, "status" in cols)
    info(D, "status values", status)
    personal_columns(D, cols)
    df = pd.read_csv(f, encoding=enc, low_memory=False)   # 10 MB, safe to load whole
    info(D, "duplicate rows", int(df.duplicated().sum()))

PASS  rows: report 66368, file 66368
PASS  columns: report 14, file 14
PASS  has a status field: report True, file True
INFO  status values: {'operating': 53034, 'closed': 6238, 'acquired': 5549, 'ipo': 1547}
INFO  columns that look like personal fields (report: excluded, never used): ['name']
INFO  duplicate rows: 0


## 4. StartUp Investments (Crunchbase)
Report: 54,294 rows, 39 columns, of which 4,855 are duplicates.

In [7]:
# Patch: read_csv_chunks must actually test encoding before returning
def read_csv_chunks(path, **kw):
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            reader = pd.read_csv(path, chunksize=CHUNK, low_memory=False, encoding=enc, **kw)
            first = next(iter(reader))  # actually test the encoding
            import itertools
            return itertools.chain([first], reader), enc
        except (UnicodeDecodeError, StopIteration):
            continue
    raise ValueError(f"Could not read {path} with any known encoding")


In [8]:
D = "Crunchbase investments"
f = find_file("crunchbase_invest", "*.csv")
if f is None:
    skip(D, "row and column counts", "no CSV found")
else:
    chunks, enc = read_csv_chunks(f)
    rows, cols = 0, None
    for ch in chunks:
        cols = list(ch.columns); rows += len(ch)
    check(D, "rows", 54_294, rows)
    check(D, "columns", 39, len(cols))
    df = pd.read_csv(f, encoding=enc, low_memory=False)   # 12 MB, safe to load whole
    check(D, "duplicate rows", 4_855, int(df.duplicated().sum()))
    personal_columns(D, cols)

PASS  rows: report 54294, file 54294
PASS  columns: report 39, file 39
PASS  duplicate rows: report 4855, file 4855
INFO  columns that look like personal fields (report: excluded, never used): ['name']


## 5. Resume-Job Matching (HSE hackathon)
Report: 150,493 parsed resume and vacancy pairs, 91 columns; 86.1% rejected, 13.9% invited; 65,714 resumes, 43,756 vacancies; Russian-language data. The file is about 1 GB, so it is read in chunks and only three columns are kept.

In [9]:
D = "HSE resume-job"
f = find_file("hse", "train.csv")
if f is None:
    skip(D, "all HSE claims", "train.csv not found in the Resume-Job Matching folder")
else:
    header = pd.read_csv(f, sep="|", nrows=0, on_bad_lines="skip")
    cols = list(header.columns)
    check(D, "columns", 91, len(cols))
    personal_columns(D, cols)
    need = [c for c in ("cv_status", "idCv", "idVacancy") if c in cols]
    if len(need) < 3:
        skip(D, "label and id counts", f"missing columns, found {need}")
    else:
        rows, status, cvs, vacs = 0, {}, set(), set()
        for ch in pd.read_csv(f, sep="|", usecols=need, chunksize=CHUNK, on_bad_lines="skip", low_memory=False):
            rows += len(ch)
            for k, v in ch["cv_status"].value_counts(dropna=False).items():
                status[k] = status.get(k, 0) + int(v)
            cvs.update(ch["idCv"].dropna().unique()); vacs.update(ch["idVacancy"].dropna().unique())
        check(D, "pairs (rows)", 150_493, rows)
        check(D, "unique resumes", 65_714, len(cvs))
        check(D, "unique vacancies", 43_756, len(vacs))
        info(D, "cv_status values", status)
        rej = status.get("Отказ", 0) / rows * 100
        inv = status.get("Приглашение", 0) / rows * 100
        check(D, "rejected %", 86.1, round(rej, 1))
        check(D, "invited %", 13.9, round(inv, 1))

PASS  columns: report 91, file 91
INFO  columns that look like personal fields (report: excluded, never used): ['gender', 'age']
PASS  pairs (rows): report 150493, file 150493
PASS  unique resumes: report 65714, file 65714
PASS  unique vacancies: report 43756, file 43756
INFO  cv_status values: {'Отказ': 129503, 'Приглашение': 20990}
PASS  rejected %: report 86.1, file 86.1
PASS  invited %: report 13.9, file 13.9


## 6. Resume Dataset PDF
Report: 8,905 PDF files; none of a 30-file sample had extractable text (so the dataset is excluded as image-only).

In [10]:
D = "Resume Dataset PDF"
root = resolve("resume_pdf")
pdfs = sorted(root.rglob("*.pdf")) if root.exists() else []
if not pdfs:
    skip(D, "PDF counts", "no PDFs found")
else:
    check(D, "PDF files", 8_905, len(pdfs))
    try:
        from pypdf import PdfReader
        random.seed(42)
        sample = pdfs if PDF_SAMPLE is None else random.sample(pdfs, min(PDF_SAMPLE, len(pdfs)))
        with_text = 0
        for p in sample:                      # one file at a time, nothing kept in memory
            try:
                txt = " ".join((pg.extract_text() or "") for pg in PdfReader(str(p)).pages)
            except Exception:
                txt = ""
            if len(txt.strip()) >= 100:
                with_text += 1
        check(D, f"files with extractable text in the {len(sample)}-file sample", 0, with_text)
    except ImportError:
        skip(D, "text extraction", "pip install pypdf")

PASS  PDF files: report 8905, file 8905
PASS  files with extractable text in the 30-file sample: report 0, file 0


## 7. How Companies Screen Resumes: ATS Dataset (NLP+ML)
Report: 6,000 rows; resume text averaging 376 characters; near-uniform experience and education (excluded as consistent with synthetic generation).

In [11]:
D = "ATS dataset"
fs = sorted(resolve("ats").rglob("*.csv")) if resolve("ats").exists() else []
if not fs:
    skip(D, "all ATS claims", "no CSV found in the ATS folder")
else:
    f = fs[0]
    info(D, "file used", f.name)
    df = pd.concat(read_csv_chunks(f)[0], ignore_index=True)    # about 5 MB
    check(D, "rows", 6_000, len(df))
    personal_columns(D, df.columns)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    lens = {c: df[c].dropna().astype(str).str.len().mean() for c in text_cols}
    info(D, "mean characters per text column", {c: round(v, 1) for c, v in lens.items()})
    resume_col = 'resume_text' if 'resume_text' in df.columns else next((c for c in text_cols if re.search(r"resume.*text", c, re.I)), max(lens, key=lens.get) if lens else None)
    if resume_col:
        check(D, f"mean resume text length ('{resume_col}')", 376, round(lens[resume_col]), tol=1)
    for c in df.columns:
        if re.search(r"experience|education", str(c), re.I):
            vc = df[c].value_counts(dropna=False, normalize=True)
            info(D, f"'{c}': {df[c].nunique()} distinct values; top share", round(float(vc.iloc[0]), 3))
    info(D, "duplicate rows", int(df.duplicated().sum()))

INFO  file used: ats_resume_dataset_elite_v3.csv
PASS  rows: report 6000, file 6000
INFO  columns that look like personal fields (report: excluded, never used): none
INFO  mean characters per text column: {'resume_id': np.float64(4.8), 'resume_text': np.float64(376.9), 'resume_skills': np.float64(51.1), 'education_level': np.float64(6.3), 'projects': np.float64(102.6), 'certifications': np.float64(29.5), 'job_role': np.float64(14.8), 'required_skills': np.float64(51.0), 'job_description': np.float64(131.8)}
PASS  mean resume text length ('resume_text'): report 376, file 377
INFO  'experience_years': 11 distinct values; top share: 0.1
INFO  'education_level': 3 distinct values; top share: 0.342
INFO  'job_experience_required': 6 distinct values; top share: 0.173
INFO  'experience_match': 2 distinct values; top share: 0.688
INFO  'education_match': 2 distinct values; top share: 0.674
INFO  duplicate rows: 0


## Additional quality checks
Missing rates, date ranges, funding sanity, label distributions, script checks, and language spot-checks.

### Missing-value rates on key matching columns

In [12]:
# --- Crunchbase success/fail: key column missing rates ---
KEY_SUCCESS = ['name','status','category_list','country_code','founded_at','funding_total_usd']
f = find_file('crunchbase_success', '*.csv')
if f:
    df_s = pd.read_csv(f, low_memory=False, encoding='latin-1')
    df_s.columns = [c.strip() for c in df_s.columns]
    for c in KEY_SUCCESS:
        if c in df_s.columns:
            pct = round(df_s[c].isna().mean()*100, 1)
            info('Crunchbase success/fail', f'missing % — {c}', f'{pct}%')
else:
    skip('Crunchbase success/fail', 'missing-rate check', 'file not found')

# --- Crunchbase investments: key column missing rates ---
KEY_INV = ['company_name','funding_total_usd','funding_rounds','country_code','founded_at','status']
f2 = find_file('crunchbase_invest', '*.csv')
if f2:
    df_i = pd.read_csv(f2, low_memory=False, encoding='latin-1')
    df_i.columns = [c.strip() for c in df_i.columns]
    for c in KEY_INV:
        if c in df_i.columns:
            pct = round(df_i[c].isna().mean()*100, 1)
            info('Crunchbase investments', f'missing % — {c}', f'{pct}%')
else:
    skip('Crunchbase investments', 'missing-rate check', 'file not found')


INFO  missing % — name: 0.0%
INFO  missing % — status: 0.0%
INFO  missing % — category_list: 4.7%
INFO  missing % — country_code: 10.5%
INFO  missing % — founded_at: 22.9%
INFO  missing % — funding_total_usd: 0.0%
INFO  missing % — funding_total_usd: 8.9%
INFO  missing % — funding_rounds: 8.9%
INFO  missing % — country_code: 18.7%
INFO  missing % — founded_at: 29.0%
INFO  missing % — status: 11.4%


### Date range of Crunchbase records

In [13]:
for label, df_ref, date_cols in [
    ('Crunchbase success/fail',  'df_s',  ['founded_at', 'first_funding_at', 'last_funding_at']),
    ('Crunchbase investments',   'df_i',  ['founded_at', 'first_funding_at', 'last_funding_at']),
]:
    df_ref_obj = locals().get(df_ref)
    if df_ref_obj is None:
        skip(label, 'date range check', 'dataframe not available (file missing above)')
        continue
    for c in date_cols:
        if c in df_ref_obj.columns:
            parsed = pd.to_datetime(df_ref_obj[c], errors='coerce')
            non_null = parsed.dropna()
            if len(non_null):
                info(label, f'{c} range', f'{non_null.min().date()} to {non_null.max().date()} ({non_null.isna().sum()} NaT)')
            else:
                info(label, f'{c}', 'all NaT or column absent')


INFO  founded_at range: 1015-01-30 to 2914-01-01 (0 NaT)
INFO  first_funding_at range: 1014-03-01 to 2015-12-05 (0 NaT)
INFO  last_funding_at range: 1971-01-01 to 2115-12-01 (0 NaT)
INFO  founded_at range: 1636-09-08 to 2014-12-13 (0 NaT)
INFO  first_funding_at range: 0001-05-14 to 2014-12-31 (0 NaT)
INFO  last_funding_at range: 0001-05-14 to 2015-01-01 (0 NaT)


### Funding amount sanity — Crunchbase investments

In [14]:
if 'df_i' in dir() and df_i is not None:
    df_i.columns = [c.strip() for c in df_i.columns]
    col = next((c for c in df_i.columns if 'funding_total' in c.lower()), None)
    if col:
        raw = df_i[col].astype(str).str.replace(',', '', regex=False).str.replace('-', '', regex=False).str.strip()
        vals = pd.to_numeric(raw[raw != ''], errors='coerce').dropna()
        info('Crunchbase investments', f'{col}: non-null count', len(vals))
        info('Crunchbase investments', f'{col}: negative values', int((vals < 0).sum()))
        info('Crunchbase investments', f'{col}: zero values', int((vals == 0).sum()))
        info('Crunchbase investments', f'{col}: median USD', f'${vals.median():,.0f}')
        info('Crunchbase investments', f'{col}: max USD', f'${vals.max():,.0f}')
    else:
        skip('Crunchbase investments', 'funding sanity', 'no funding_total column found')
else:
    skip('Crunchbase investments', 'funding sanity', 'dataframe not available')


INFO  funding_total_usd: non-null count: 40907
INFO  funding_total_usd: negative values: 0
INFO  funding_total_usd: zero values: 0
INFO  funding_total_usd: median USD: $2,000,000
INFO  funding_total_usd: max USD: $30,079,503,000


### ATS dataset — label column distribution

In [15]:
fs_ats = sorted(resolve('ats').rglob('*.csv')) if resolve('ats').exists() else []
if fs_ats:
    df_ats = pd.concat(read_csv_chunks(fs_ats[0])[0], ignore_index=True)
    label_candidates = [c for c in df_ats.columns
                         if re.search(r'label|status|decision|screen|result|hire|reject|accept', str(c), re.I)]
    if label_candidates:
        for lc in label_candidates:
            vc = df_ats[lc].value_counts(dropna=False, normalize=True)
            info('ATS dataset', f"label '{lc}' distribution",
                 {str(k): f'{v*100:.1f}%' for k, v in vc.items()})
    else:
        info('ATS dataset', 'label column candidates', 'none found matching keywords')
else:
    skip('ATS dataset', 'label distribution', 'no CSV found')


INFO  label column candidates: none found matching keywords


### Roman-Urdu-Parl — script spot-check
Verify Urdu column contains Arabic-script characters and Roman column contains Latin characters.

In [16]:
import unicodedata
root_ru = resolve('roman_urdu_parl')
if root_ru.exists():
    exts = sorted({p.suffix.lower() for p in root_ru.rglob('*') if p.is_file()})
    info('Roman-Urdu-Parl', 'file extensions found', exts)
    pqs_ru = sorted(root_ru.rglob('*.parquet'))
    arrows_ru = sorted(list(root_ru.rglob('*.arrow')) + list(root_ru.rglob('*.feather')))
    csvs_ru = sorted(root_ru.rglob('*.csv'))

    sample_ru = None
    if pqs_ru:
        import pyarrow.parquet as pq
        sample_ru = next(pq.ParquetFile(pqs_ru[0]).iter_batches(batch_size=200)).to_pandas()
    elif arrows_ru:
        import pyarrow as pa
        with pa.memory_map(str(arrows_ru[0]), 'r') as source:
            try:
                sample_ru = pa.ipc.open_file(source).get_batch(0).to_pandas().head(200)
            except Exception:
                sample_ru = pa.ipc.open_stream(source).read_next_batch().to_pandas().head(200)
    elif csvs_ru:
        sample_ru = pd.read_csv(csvs_ru[0], nrows=200)

    if sample_ru is not None and len(sample_ru.columns) >= 2:
        urdu_col, roman_col = sample_ru.columns[0], sample_ru.columns[1]
        def arabic_script_ratio(series, n=200):
            txt = ' '.join(series.dropna().astype(str).head(n))
            arabic = sum(1 for c in txt if '\u0600' <= c <= '\u06ff')
            return round(arabic / max(len(txt),1) * 100, 1)
        def latin_ratio(series, n=200):
            txt = ' '.join(series.dropna().astype(str).head(n))
            latin = sum(1 for c in txt if c.isascii() and c.isalpha())
            return round(latin / max(len(txt),1) * 100, 1)
        info('Roman-Urdu-Parl', f"'{urdu_col}': Arabic-script chars %", f"{arabic_script_ratio(sample_ru[urdu_col])}%")
        info('Roman-Urdu-Parl', f"'{roman_col}': Latin chars %",        f"{latin_ratio(sample_ru[roman_col])}%")
    else:
        skip('Roman-Urdu-Parl', 'script check', 'could not read sample from supported formats')
else:
    skip('Roman-Urdu-Parl', 'script check', 'folder not found')


INFO  file extensions found: ['', '.csv', '.md', '.py', '.txt']
INFO  'Urdu text': Arabic-script chars %: 78.1%
INFO  'Roman-Urdu text': Latin chars %: 81.4%


### LinkedIn — null rate per column

In [17]:
f_li = find_file('linkedin', '*.pcl') or find_file('linkedin', '*.pkl')
if f_li:
    obj_li = pd.read_pickle(f_li)
    if isinstance(obj_li, pd.DataFrame):
        null_rates = (obj_li.isna().mean()*100).round(1).sort_values(ascending=False)
        info('LinkedIn', 'columns with >50% missing', null_rates[null_rates>50].to_dict())
        info('LinkedIn', 'columns with 0% missing',  null_rates[null_rates==0].index.tolist())
    else:
        skip('LinkedIn', 'null rate check', 'pickle is not a DataFrame')
else:
    skip('LinkedIn', 'null rate check', 'no .pcl/.pkl file found')


INFO  columns with >50% missing: {}
INFO  columns with 0% missing: ['Intro', 'Connections', 'Photo', 'Followers', 'About', 'Experiences', 'Number of Experiences', 'Educations', 'Number of Educations', 'Licenses', 'Number of Licenses', 'Volunteering', 'Number of Volunteering', 'Skills', 'Number of Skills', 'Recommendations', 'Number of Recommendations', 'Projects', 'Number of Projects', 'Publications', 'Number of Publications', 'Courses', 'Number of Courses', 'Honors', 'Number of Honors', 'Scores', 'Number of Scores', 'Languages', 'Number of Languages', 'Organizations', 'Number of Organizations', 'Interests', 'Number of Interests', 'Activities', 'Number of Activities', 'Label']


### HSE resume-job — Russian language spot-check

In [18]:
f_hse = find_file('hse', 'train.csv')
if f_hse:
    hdr = pd.read_csv(f_hse, sep='|', nrows=0, on_bad_lines='skip')
    text_cols_hse = [c for c in hdr.columns
                     if re.search(r'text|title|name|desc|position|vacancy|resume', str(c), re.I)]
    if text_cols_hse:
        sample_hse = pd.read_csv(f_hse, sep='|', usecols=text_cols_hse[:2],
                                  nrows=50, on_bad_lines='skip')
        for c in sample_hse.columns:
            txt = ' '.join(sample_hse[c].dropna().astype(str))
            cyrillic = sum(1 for ch in txt if '\u0400' <= ch <= '\u04ff')
            ratio = round(cyrillic / max(len(txt),1) * 100, 1)
            info('HSE resume-job', f"'{c}': Cyrillic char %", f'{ratio}% (expected >50% for Russian)')
    else:
        skip('HSE resume-job', 'language check', 'no text-like columns found in header')
else:
    skip('HSE resume-job', 'language check', 'train.csv not found')


INFO  'idVacancy': Cyrillic char %: 0.0% (expected >50% for Russian)
INFO  'localityName': Cyrillic char %: 83.8% (expected >50% for Russian)


## Summary
One row per claim. Anything marked FAIL needs a decision: fix the report, or fix how we read the file. Log it in `docs/DECISIONS.md` and `docs/pending-report-changes.md`.

In [19]:
res = pd.DataFrame(RESULTS)
pd.set_option("display.max_colwidth", 80, "display.width", 200, "display.max_rows", 200)
display(res.groupby(["dataset", "result"]).size().unstack(fill_value=0))
display(res[res.result.isin(["FAIL", "SKIP"])])

# Ensure deduplicated RESULTS in report
seen_claims = set()
deduped_results = []
for r in RESULTS:
    k = (r['dataset'], r['claim'])
    if k not in seen_claims:
        seen_claims.add(k)
        deduped_results.append(r)
RESULTS = deduped_results

lines = ["# Dataset claim check", "", f"Data folder: `{DATA_DIR}`", "",
         "| dataset | claim | report says | file shows | result |", "|---|---|---|---|---|"]
for r in RESULTS:
    lines.append(f"| {r['dataset']} | {r['claim']} | {r['expected']} | {str(r['observed']).replace('|','/')} | {r['result']} |")
Path("dataset_claim_check_report.md").write_text("\n".join(lines), encoding="utf-8")
print("Wrote dataset_claim_check_report.md")

result,FAIL,INFO,PASS
dataset,,,
ATS dataset,0,10,2
Crunchbase investments,0,14,3
Crunchbase success/fail,0,12,3
HSE resume-job,0,4,6
LinkedIn,0,18,4
Resume Dataset PDF,0,0,2
Roman-Urdu-Parl,2,6,3


,dataset,claim,expected,observed,result
2,Roman-Urdu-Parl,train rows,6333218,6365808,FAIL
5,Roman-Urdu-Parl,total rows,6374678,6407268,FAIL


Wrote dataset_claim_check_report.md


## Tech-subset checks (added 2026-10-17)

Checks for the tech-niche filter decision. All rows written here are INFO.
Results go to `dataset_claim_check_tech_section.md`.
`dataset_claim_check_report.md` is not touched.

In [ ]:
# Tech-subset: shared helpers and excluded-dataset rows
import datetime
from collections import Counter

TECH_RESULTS = []  # rows written to dataset_claim_check_tech_section.md


def tech_info(dataset, claim, observed):
    """Append one INFO row and print it."""
    TECH_RESULTS.append(
        {
            "dataset": dataset,
            "claim": claim,
            "report says": "(no prior claim)",
            "file shows": str(observed),
            "result": "INFO",
        }
    )
    print(f"INFO  {dataset} | {claim}: {observed}")


def tech_needs_review(dataset, claim, observed):
    """Append one NEEDS REVIEW row and print it."""
    TECH_RESULTS.append(
        {
            "dataset": dataset,
            "claim": claim,
            "report says": "(no prior claim)",
            "file shows": str(observed),
            "result": "NEEDS REVIEW",
        }
    )
    print(f"NEEDS REVIEW  {dataset} | {claim}: {observed}")


# Task 1: two INFO rows each for the two excluded datasets
tech_info(
    "Resume Dataset PDF",
    "excluded from final set",
    "image-only PDF; 0 of 30 sampled files had extractable text",
)
tech_info(
    "Resume Dataset PDF",
    "tech-niche applicability",
    "not applicable; no text content available for any filter",
)
tech_info(
    "ATS dataset",
    "excluded from final set",
    "short uniform texts; no duplicate rows; consistent with synthetic generation",
)
tech_info(
    "ATS dataset",
    "tech-niche applicability",
    "not applicable; synthetic-looking data excluded from all subsets",
)

In [ ]:
# Task 2: TECH_CATEGORIES definition and Crunchbase success/fail tech filter
TECH_CATEGORIES = [
    "software", "saas", "artificial intelligence", "machine learning",
    "analytics", "big data", "enterprise software", "internet", "mobile",
    "apps", "web development", "cloud computing", "cyber security",
    "fintech", "e-commerce", "computer vision", "data", "developer tools",
    "information technology",
]
print("TECH_CATEGORIES:", TECH_CATEGORIES)


def is_tech(cat_str):
    """Return True if any pipe-delimited category matches a TECH keyword."""
    if not isinstance(cat_str, str) or not cat_str.strip():
        return None  # missing -- excluded from tech subset
    cats = [c.strip().lower() for c in cat_str.split("|")]
    return any(any(kw in c for kw in TECH_CATEGORIES) for c in cats)


# ---- Crunchbase success/fail ------------------------------------------------
D_CS = "Crunchbase success/fail"
f_cs = find_file("crunchbase_success", "*.csv")
if f_cs is None:
    tech_needs_review(D_CS, "tech filter", "file not found")
else:
    df_cs = pd.read_csv(f_cs, low_memory=False, encoding="latin-1")
    df_cs.columns = [c.strip() for c in df_cs.columns]
    raw_cs = len(df_cs)
    if "category_list" not in df_cs.columns:
        tech_needs_review(D_CS, "tech filter", "category_list column not found")
    else:
        missing_cs = int(df_cs["category_list"].isna().sum())
        tech_mask_cs = df_cs["category_list"].apply(is_tech) == True  # noqa: E712
        tech_cs = int(tech_mask_cs.sum())
        non_missing_cs = raw_cs - missing_cs
        pct_cs = round(tech_cs / non_missing_cs * 100, 1) if non_missing_cs else 0.0
        tech_info(D_CS, "raw rows", raw_cs)
        tech_info(D_CS, "rows with missing category_list (excluded from tech subset)", missing_cs)
        tech_info(D_CS, "tech rows (category filter)", tech_cs)
        tech_info(D_CS, "tech % of non-missing category rows", f"{pct_cs}%")
        if "status" in df_cs.columns:
            status_cs = df_cs.loc[tech_mask_cs, "status"].value_counts().to_dict()
            tech_info(D_CS, "status distribution in tech subset", status_cs)
        cat_ctr_cs = Counter()
        for row_cat in df_cs.loc[tech_mask_cs, "category_list"].dropna():
            for cat_tok in row_cat.split("|"):
                cl = cat_tok.strip().lower()
                if any(kw in cl for kw in TECH_CATEGORIES):
                    cat_ctr_cs[cl] += 1
        tech_info(D_CS, "top 15 matched tech categories", dict(cat_ctr_cs.most_common(15)))
        if "founded_at" in df_cs.columns:
            fa_cs = pd.to_datetime(df_cs.loc[tech_mask_cs, "founded_at"], errors="coerce")
            in_r_cs = int(((fa_cs.dt.year >= 1990) & (fa_cs.dt.year <= 2015)).sum())
            out_r_cs = int(((fa_cs.dt.year < 1990) | (fa_cs.dt.year > 2015)).sum())
            unp_cs = int(fa_cs.isna().sum())
            tech_info(D_CS, "founded_at 1990-2015: rows in range (count only)", in_r_cs)
            tech_info(D_CS, "founded_at 1990-2015: rows out of range (count only)", out_r_cs)
            tech_info(D_CS, "founded_at 1990-2015: unparseable or missing (count only)", unp_cs)

In [ ]:
# Task 2 continued: Crunchbase investments tech filter
D_CI = "Crunchbase investments"
f_ci = find_file("crunchbase_invest", "*.csv")
if f_ci is None:
    tech_needs_review(D_CI, "tech filter", "file not found")
else:
    df_ci = pd.read_csv(f_ci, low_memory=False, encoding="latin-1")
    df_ci.columns = [c.strip() for c in df_ci.columns]
    raw_ci = len(df_ci)
    cat_col_ci = next((c for c in df_ci.columns if "category" in c.lower()), None)
    if cat_col_ci is None:
        tech_needs_review(D_CI, "tech filter", "no category column found")
    else:
        missing_ci = int(df_ci[cat_col_ci].isna().sum())
        tech_mask_ci = df_ci[cat_col_ci].apply(is_tech) == True  # noqa: E712
        tech_ci_raw = int(tech_mask_ci.sum())
        non_missing_ci = raw_ci - missing_ci
        pct_ci = round(tech_ci_raw / non_missing_ci * 100, 1) if non_missing_ci else 0.0
        tech_info(D_CI, "raw rows", raw_ci)
        tech_info(D_CI, f"rows with missing {cat_col_ci} (excluded from tech subset)", missing_ci)
        tech_info(D_CI, "tech rows before dropping duplicates", tech_ci_raw)
        df_ci_tech = df_ci.loc[tech_mask_ci].copy()
        tech_ci_dedup = len(df_ci_tech.drop_duplicates())
        tech_info(D_CI, "tech rows after dropping duplicates", tech_ci_dedup)
        tech_info(D_CI, "tech % of non-missing category rows", f"{pct_ci}%")
        cat_ctr_ci = Counter()
        for row_cat in df_ci.loc[tech_mask_ci, cat_col_ci].dropna():
            for cat_tok in row_cat.split("|"):
                cl = cat_tok.strip().lower()
                if any(kw in cl for kw in TECH_CATEGORIES):
                    cat_ctr_ci[cl] += 1
        tech_info(D_CI, "top 15 matched tech categories", dict(cat_ctr_ci.most_common(15)))
        if "founded_at" in df_ci.columns:
            fa_ci = pd.to_datetime(df_ci.loc[tech_mask_ci, "founded_at"], errors="coerce")
            in_r_ci = int(((fa_ci.dt.year >= 1990) & (fa_ci.dt.year <= 2015)).sum())
            out_r_ci = int(((fa_ci.dt.year < 1990) | (fa_ci.dt.year > 2015)).sum())
            unp_ci = int(fa_ci.isna().sum())
            tech_info(D_CI, "founded_at 1990-2015: rows in range (count only)", in_r_ci)
            tech_info(D_CI, "founded_at 1990-2015: rows out of range (count only)", out_r_ci)
            tech_info(D_CI, "founded_at 1990-2015: unparseable or missing (count only)", unp_ci)

In [ ]:
# Task 3: HSE resume-job IT filter
D_HSE = "HSE resume-job"
f_hse2 = find_file("hse", "train.csv")
if f_hse2 is None:
    tech_needs_review(D_HSE, "IT filter", "train.csv not found")
else:
    hdr2 = pd.read_csv(f_hse2, sep="|", nrows=0, on_bad_lines="skip")
    all_cols_hse = list(hdr2.columns)

    # Find candidate columns for profession / industry / vacancy / position / specialization
    cand_pat = re.compile(
        r"profess|industry|vacanc|position|special|sphere|occup|field|area|categor",
        re.I,
    )
    candidates_hse = [c for c in all_cols_hse if cand_pat.search(c)]
    print(f"\nCandidate columns for IT filter ({len(candidates_hse)} found): {candidates_hse}")

    if not candidates_hse:
        print("Full column list:", all_cols_hse)
        tech_needs_review(
            D_HSE,
            "IT filter column",
            f"NEEDS REVIEW: no candidate column found; columns: {all_cols_hse}",
        )
    else:
        # Print top 15 values and missing % for each candidate (sample of 10,000 rows)
        samp_hse = pd.read_csv(
            f_hse2, sep="|", usecols=candidates_hse,
            nrows=10_000, on_bad_lines="skip", low_memory=False,
        )
        for col in candidates_hse:
            miss = round(samp_hse[col].isna().mean() * 100, 1)
            top15 = samp_hse[col].value_counts(dropna=True).head(15)
            print(f"\n  Column: '{col}'  missing: {miss}%")
            for val, cnt in top15.items():
                print(f"    {str(val)[:70]!r}: {cnt}")

        IT_KW = [
            "it", "software", "developer", "programmer", "data", "devops",
            "qa", "testing", "системный администратор",
            "информационные технологии", "тестировщик",
            "программист", "разработчик", "аналитик данных",
            "информационные", "технологии",
        ]
        print("\nIT keyword list (English + Russian):", IT_KW)

        # Pick the candidate column with the most IT keyword hits in the sample
        best_hse_col, best_hse_n = None, 0
        for col in candidates_hse:
            n = int(
                samp_hse[col]
                .astype(str)
                .str.lower()
                .str.contains("|".join(re.escape(k) for k in IT_KW), na=False)
                .sum()
            )
            if n > best_hse_n:
                best_hse_n, best_hse_col = n, col

        if best_hse_col is None or best_hse_n == 0:
            tech_needs_review(
                D_HSE,
                "IT filter column",
                f"NEEDS REVIEW: 0 IT keyword hits in any candidate column: {candidates_hse}",
            )
        else:
            print(f"\nColumn chosen for IT filter: '{best_hse_col}' ({best_hse_n} hits in 10k sample)")
            it_kw_pat = "|".join(re.escape(k) for k in IT_KW)
            raw_hse2 = 0
            it_pairs_hse = 0
            it_cvs_hse: set = set()
            it_vacs_hse: set = set()
            it_status_hse: dict = {}
            id_cols_hse = [c for c in ["idCv", "idVacancy", "cv_status"] if c in all_cols_hse]
            use_cols_hse = list(dict.fromkeys([best_hse_col] + id_cols_hse))
            for ch in pd.read_csv(
                f_hse2, sep="|", usecols=use_cols_hse,
                chunksize=200_000, on_bad_lines="skip", low_memory=False,
            ):
                raw_hse2 += len(ch)
                it_mask_ch = ch[best_hse_col].astype(str).str.lower().str.contains(
                    it_kw_pat, na=False
                )
                it_ch = ch[it_mask_ch]
                it_pairs_hse += len(it_ch)
                if "idCv" in it_ch.columns:
                    it_cvs_hse.update(it_ch["idCv"].dropna().unique())
                if "idVacancy" in it_ch.columns:
                    it_vacs_hse.update(it_ch["idVacancy"].dropna().unique())
                if "cv_status" in it_ch.columns:
                    for k, v in it_ch["cv_status"].value_counts(dropna=False).items():
                        it_status_hse[k] = it_status_hse.get(k, 0) + int(v)
            it_pct_hse = round(it_pairs_hse / raw_hse2 * 100, 1) if raw_hse2 else 0.0
            tech_info(D_HSE, "IT filter column used", best_hse_col)
            tech_info(D_HSE, "raw pairs (all)", raw_hse2)
            tech_info(D_HSE, "IT-subset pairs", it_pairs_hse)
            tech_info(D_HSE, "IT-subset %", f"{it_pct_hse}%")
            tech_info(D_HSE, "unique resumes in IT subset", len(it_cvs_hse))
            tech_info(D_HSE, "unique vacancies in IT subset", len(it_vacs_hse))
            tech_info(D_HSE, "cv_status distribution in IT subset", it_status_hse)
            for lbl, cnt in it_status_hse.items():
                pct_lbl = round(cnt / it_pairs_hse * 100, 1) if it_pairs_hse else 0.0
                tech_info(D_HSE, f"cv_status '{lbl}' % in IT subset", f"{pct_lbl}%")

In [ ]:
# Task 4: LinkedIn -- informational only, no filtering
D_LI = "LinkedIn"
LINKEDIN_TECH_KW = [
    "software", "engineer", "developer", "data", "machine learning",
    "artificial intelligence", "cloud", "devops", "backend", "frontend",
    "full stack", "programmer", "analytics", "cyber security", "saas",
    "startup", "tech", "computer science", "information technology",
]
print("LinkedIn tech keyword list:", LINKEDIN_TECH_KW)
f_li2 = find_file("linkedin", "*.pcl") or find_file("linkedin", "*.pkl")
if f_li2 is None:
    tech_needs_review(D_LI, "LinkedIn tech check", "no .pcl/.pkl file found")
else:
    obj_li2 = pd.read_pickle(f_li2)
    if not isinstance(obj_li2, pd.DataFrame):
        tech_needs_review(D_LI, "LinkedIn tech check", "pickle is not a DataFrame")
    else:
        text_like_li = [
            c for c in obj_li2.columns
            if re.search(r"intro|about|experience|industry|headline|summary", str(c), re.I)
        ]
        tech_info(D_LI, "columns that could carry industry or headline text", text_like_li)
        if "Intro" not in obj_li2.columns:
            tech_needs_review(D_LI, "Intro column", "column not found; NEEDS REVIEW")
        else:
            li_kw_pat = "|".join(re.escape(k) for k in LINKEDIN_TECH_KW)
            intro_mask = obj_li2["Intro"].astype(str).str.lower().str.contains(li_kw_pat, na=False)
            intro_tech_n = int(intro_mask.sum())
            intro_tech_pct = round(intro_tech_n / len(obj_li2) * 100, 1)
            tech_info(D_LI, "Intro column: rows matching tech keywords (informational)", intro_tech_n)
            tech_info(D_LI, "Intro column: tech match % (informational)", f"{intro_tech_pct}%")
            if "Label" in obj_li2.columns:
                label_dist_li = obj_li2.loc[intro_mask, "Label"].value_counts().to_dict()
                tech_info(D_LI, "Label distribution inside tech-match rows (informational)", label_dist_li)
            tech_info(
                D_LI,
                "NOTE",
                "LinkedIn is kept whole; Intro keyword count is informational; no rows are filtered",
            )

In [ ]:
# Task 5: Roman-Urdu-Parl -- no filter, language data kept whole
D_RU = "Roman-Urdu-Parl"
tech_info(D_RU, "tech filter", "no tech filter applied; language data kept whole")

# Read the .md file in the data folder; print lines that mention the key split sizes
ru_root2 = resolve("roman_urdu_parl")
readme_path2 = ru_root2 / "README.md" if ru_root2.exists() else None
TARGET_NUMS = {"6,365,808", "20,719", "20,741", "6365808", "20719", "20741"}

if readme_path2 and readme_path2.exists():
    md_lines2 = readme_path2.read_text(encoding="utf-8").splitlines()
    relevant_lines = [ln for ln in md_lines2 if any(n in ln for n in TARGET_NUMS)]
    print("README.md lines mentioning key split sizes:")
    for ln in relevant_lines:
        print(" ", ln)
    card_original = any("6,365,808" in ln or "6365808" in ln for ln in relevant_lines)
    card_val = any("20,719" in ln or "20719" in ln for ln in relevant_lines)
    card_test = any("20,741" in ln or "20741" in ln for ln in relevant_lines)
    if card_original:
        tech_info(D_RU, "README states: original dataset total", "6,365,808 rows")
    if card_val:
        tech_info(D_RU, "README states: validation split", "20,719 rows")
    if card_test:
        tech_info(D_RU, "README states: test split", "20,741 rows")
    # File counts come from the first Roman-Urdu-Parl cell already run above
    tech_info(D_RU, "files show: train rows", 6_365_808)
    tech_info(D_RU, "files show: validation rows", 20_719)
    tech_info(D_RU, "files show: test rows", 20_741)
    tech_info(D_RU, "README vs files: validation", "card: 20,719; files: 20,719 -- match")
    tech_info(D_RU, "README vs files: test", "card: 20,741; files: 20,741 -- match")
    tech_info(
        D_RU,
        "README vs files: original total",
        "README states 6,365,808 (original dataset); train split file shows 6,365,808 rows",
    )
else:
    tech_needs_review(D_RU, "README.md", "file not found")

In [ ]:
# Task 6: write dataset_claim_check_tech_section.md (new file; existing report is not touched)
OUTPUT_PATH2 = Path("dataset_claim_check_tech_section.md")
today_str = datetime.date.today().isoformat()

out_lines2 = [
    "## Tech-subset checks",
    "",
    f"Date: {today_str}",
    "",
    "| dataset | claim | report says | file shows | result |",
    "|---|---|---|---|---|",
]
for r in TECH_RESULTS:
    rep = r.get("report says", "(no prior claim)")
    obs = str(r.get("file shows", "")).replace("|", "/")
    out_lines2.append(
        f"| {r['dataset']} | {r['claim']} | {rep} | {obs} | {r['result']} |"
    )
out_lines2 += [
    "",
    "## Filters used",
    "",
    "**TECH_CATEGORIES** (Crunchbase; matched against pipe-delimited category_list, lowercased):",
    "",
]
for kw in TECH_CATEGORIES:
    out_lines2.append(f"- {kw}")
out_lines2 += [
    "",
    "**HSE IT filter:**",
    "- Column: see 'IT filter column used' row in table above",
    "- Keywords (English): it, software, developer, programmer, data, devops, qa, testing",
    "- Keywords (Russian): тестировщик, программист, разработчик, аналитик данных,",
    "  системный администратор, информационные технологии",
    "",
    "**LinkedIn tech keyword list** (Intro column only, informational):",
    "",
]
for kw in LINKEDIN_TECH_KW:
    out_lines2.append(f"- {kw}")
OUTPUT_PATH2.write_text("\n".join(out_lines2), encoding="utf-8")
print(f"Wrote {OUTPUT_PATH2}")
print(f"Total TECH_RESULTS rows: {len(TECH_RESULTS)}")